# 5.9 Python

Chapter 5 develops the theory of continuous distributions. This section uses Python to work with three foundational families — Uniform, Normal, and Exponential — evaluating their PDFs and CDFs via `scipy.stats`, plotting them, and building intuition through simulation. We also verify the universality of the Uniform by transforming Uniform draws into a Logistic distribution, and simulate a Poisson process by generating exponential interarrival times.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Uniform, Normal, and Exponential Distributions

For continuous distributions, `scipy.stats` objects expose `.pdf` for the probability density function and `.cdf` for the cumulative distribution function — the continuous analogues of `.pmf` and `.cdf` for discrete distributions.

### Uniform

The $\mathrm{Unif}(a, b)$ distribution is parameterized in `scipy.stats` as `st.uniform(loc=a, scale=b-a)`, where `loc` sets the lower endpoint and `scale` sets the width. The default with no arguments is $\mathrm{Unif}(0, 1)$.

In [ ]:
# Unif(0, 1) PDF and CDF at x = 0.5
print(st.uniform.pdf(0.5))   # PDF = 1 everywhere on (0,1)
print(st.uniform.cdf(0.5))   # CDF = 0.5 at the midpoint

# Unif(1, 3): loc=1, scale=2
a, b = 1, 3
print(st.uniform.pdf(2, loc=a, scale=b - a))   # PDF = 1/(b-a) = 0.5
print(st.uniform.cdf(2, loc=a, scale=b - a))   # CDF = (2-1)/(3-1) = 0.5

### Normal

For $X \sim N(\mu, \sigma^2)$, use `st.norm(loc=mu, scale=sigma)`, where `scale` is the **standard deviation** $\sigma$, not the variance $\sigma^2$. Confusing the two is a common source of error: to work with $N(10, 3)$ — variance 3, standard deviation $\sqrt{3}$ — pass `scale=np.sqrt(3)`, not `scale=3`.

In [ ]:
# Standard Normal N(0, 1): PDF and CDF at x = 0
print(st.norm.pdf(0))   # 1/sqrt(2*pi) ≈ 0.3989
print(st.norm.cdf(0))   # 0.5 by symmetry

# N(10, 3): mean 10, variance 3, std dev sqrt(3)
mu, var = 10, 3
print(st.norm.cdf(12, loc=mu, scale=np.sqrt(var)))   # P(X <= 12)

### Exponential

For $X \sim \mathrm{Expo}(\lambda)$, the mean is $1/\lambda$. `scipy.stats` parameterizes the Exponential by its mean rather than its rate: `st.expon(scale=1/lam)`. Passing the rate $\lambda$ directly as `scale` would give the wrong distribution, so always take the reciprocal.

In [ ]:
lam = 5
print(st.expon.pdf(0.2, scale=1 / lam))   # PDF at x=0.2 for Expo(5)
print(st.expon.cdf(0.2, scale=1 / lam))   # P(X <= 0.2) for Expo(5)

# Expo(1) is the default
print(st.expon.pdf(1))   # PDF at x=1 for Expo(1): e^{-1} ≈ 0.3679

## Generating Random Draws and Location-Scale Transformations

NumPy's random generator provides direct methods for each family. Because the Normal family is closed under location-scale transformations, there are two equivalent ways to draw from $N(\mu, \sigma^2)$: pass the parameters directly, or draw from $N(0,1)$ and shift-and-scale the result. Both produce the same distribution.

In [ ]:
mu, sigma = 1, 2

# Direct parameterization
draws_direct = rng.normal(loc=mu, scale=sigma, size=5)

# Location-scale transformation from N(0,1)
draws_shifted = mu + sigma * rng.normal(size=5)

print("Direct:    ", np.round(draws_direct, 4))
print("Shifted:   ", np.round(draws_shifted, 4))

The outputs differ because they use independent draws from the generator, but both sequences are realizations from $N(1, 4)$. The location-scale approach is useful when working symbolically: it makes explicit that any $N(\mu, \sigma^2)$ random variable is just a shifted and scaled version of the standard Normal.

The same principle applies to the Uniform and Exponential. A $\mathrm{Unif}(a, b)$ draw is $a + (b-a)U$ for $U \sim \mathrm{Unif}(0,1)$, and an $\mathrm{Expo}(\lambda)$ draw is $-\log(U)/\lambda$ for the same $U$.

## Plotting PDFs

To plot a continuous density, evaluate it on a fine grid of points and connect them with a line. `np.linspace(a, b, n)` produces `n` evenly spaced values from `a` to `b`; choosing `n` large enough (e.g., 500 or 1000) makes the curve look smooth. The standard Normal PDF extends practically from $-3$ to $3$, so that is a natural plotting range.

In [ ]:
x = np.linspace(-3, 3, 1000)
y = st.norm.pdf(x)

fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(x=x, y=y, ax=ax)
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title("Standard Normal PDF")
plt.tight_layout()
plt.show()

The same pattern works for any `scipy.stats` distribution. Below we plot the PDFs of $\mathrm{Unif}(0, 1)$ and $\mathrm{Expo}(1)$ side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

x_unif = np.linspace(-0.5, 1.5, 500)
sns.lineplot(x=x_unif, y=st.uniform.pdf(x_unif), ax=axes[0])
axes[0].set_xlabel("x")
axes[0].set_ylabel("Density")
axes[0].set_title("Unif(0, 1) PDF")

x_exp = np.linspace(0, 1, 500)
sns.lineplot(x=x_exp, y=st.expon.pdf(x_exp), ax=axes[1])
axes[1].set_xlabel("x")
axes[1].set_ylabel("Density")
axes[1].set_title("Expo(1) PDF")

plt.tight_layout()
plt.show()

## Universality of the Uniform

Example 5.3.4 establishes that if $U \sim \mathrm{Unif}(0, 1)$, then the transformation

$$X = \log\!\left(\frac{U}{1 - U}\right)$$

follows a Logistic distribution. This is an instance of the universality of the Uniform: applying the quantile function of any continuous distribution to a $\mathrm{Unif}(0,1)$ random variable produces a draw from that distribution.

We can verify the result by generating $10^4$ Uniform draws, applying the transformation, and comparing the resulting histogram to the Logistic PDF.

In [ ]:
u = rng.uniform(size=10**4)
x = np.log(u / (1 - u))

x_grid = np.linspace(-8, 8, 500)
logistic_pdf = st.logistic.pdf(x_grid)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(x, stat="density", bins=80, ax=ax, label="Simulated")
sns.lineplot(x=x_grid, y=logistic_pdf, ax=ax, label="Logistic PDF")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title(r"$\log(U / (1-U))$ vs. Logistic PDF")
ax.legend()
plt.tight_layout()
plt.show()

The histogram closely tracks the theoretical Logistic curve. Increasing the number of draws or reducing the bin width would bring them into even tighter agreement.

## Poisson Process Simulation

A Poisson process with rate $\lambda$ has the property that the times between consecutive arrivals — called interarrival times — are i.i.d. $\mathrm{Expo}(\lambda)$ random variables. To simulate the first $n$ arrivals, we generate $n$ interarrival times and convert them to arrival times using a cumulative sum.

In [ ]:
n = 50
lam = 10

interarrival = rng.exponential(scale=1 / lam, size=n)
arrival_times = np.cumsum(interarrival)

print("First 10 arrival times:")
print(np.round(arrival_times[:10], 4))

With rate $\lambda = 10$, the mean interarrival time is $1/10 = 0.1$, so we expect about 10 arrivals per unit of time. Plotting the simulated arrival times as marks on a timeline makes this density of events visible.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2))
sns.rugplot(x=arrival_times, ax=ax, height=0.6)
ax.set_xlabel("Time")
ax.set_yticks([])
ax.set_title(f"Simulated Poisson Process Arrivals  (λ = {lam}, n = {n})")
plt.tight_layout()
plt.show()

The tick marks are not evenly spaced — they cluster and spread in an irregular, memoryless fashion, which is the characteristic signature of a Poisson process. The 50 arrivals span a time interval of roughly $50/10 = 5$ units, consistent with the expected rate.